# Unmasked Network Linear-Systems Analysis

This notebook extends the Schur/eigenmode analysis with explicit intermediate steps.

It assumes your original `mij_matrix.csv` has:

\[
\text{rows}=\text{outgoing/source nodes}, \qquad \text{columns}=\text{receiver/target nodes}.
\]

The notebook converts that file into the standard column-vector state convention:

\[
x_{t+1}=Mx_t,\qquad M_{ij}=\text{effect of source }j\text{ on receiver }i.
\]

Therefore:

\[
M=\texttt{df\_source\_receiver.to\_numpy().T}.
\]

The analyses below are:

1. **Structural path analysis** using the matrix power series \(M, M^2, M^3,\ldots\).
2. **Controllability Gramian analysis** for state diffusion.
3. **Frequency response analysis** using

\[
G(s)=C(sI-A)^{-1}B.
\]

Most reusable logic lives in `network_linear_systems_tools_script.py`.

## 0. Notebook Controls

Change these settings before running the notebook.

The default analysis preserves the original weight scale. You can set `NORMALIZATION = "spectral_radius"` if you want equal-gain analysis.

In [ ]:
from pathlib import Path

CSV_PATH = Path("mij_matrix.csv")

# Matrix normalization:
#   "none"            -> preserve absolute connectivity scale
#   "column_l1"       -> normalize each source column after orientation correction
#   "spectral_radius" -> scale M so rho(M)=TARGET_SPECTRAL_RADIUS
NORMALIZATION = "none"
TARGET_SPECTRAL_RADIUS = 1.0

# Matrix power/path analysis
MAX_PATH_LENGTH = 6
PATH_ALPHA = 1.0
TOP_N_PRINT = 12

# Discrete-time finite-horizon controllability Gramian
GRAMIAN_HORIZON = 25

# Inputs and outputs:
#   None means all nodes.
#   Or use labels, e.g. DRIVER_NODES = ["EC LII Stellate", "CA3 Pyramidal"]
DRIVER_NODES = None
OUTPUT_NODES = None

# Continuous-time frequency response.
# A = M - shift*I is used by default so that transfer-function analysis is stable.
CONTINUOUS_SHIFT_MARGIN = 0.25
FREQ_MIN = 1e-3
FREQ_MAX = 1e2
FREQ_POINTS = 250

print("Controls loaded.")
print(f"CSV_PATH: {CSV_PATH}")
print(f"NORMALIZATION: {NORMALIZATION}")
print(f"MAX_PATH_LENGTH: {MAX_PATH_LENGTH}")
print(f"GRAMIAN_HORIZON: {GRAMIAN_HORIZON}")

## 1. Imports and External Utility Module

The companion file `network_linear_systems_tools_script.py` contains reusable functions for:

- loading and orienting the matrix,
- computing matrix powers,
- extracting top source→receiver path entries,
- controllability Gramians,
- stable continuous-time conversion,
- transfer functions and frequency responses.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import linalg

# Make sure the local module can be imported from the notebook directory.
sys.path.insert(0, str(Path(".").resolve()))

import network_linear_systems_tools_script as nlt

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

print("Imported network_linear_systems_tools_script as nlt")
print("Current working directory:", Path.cwd())

## 2. Load Matrix and Unmask the Orientation Step

The raw CSV is interpreted as source→receiver.

The state matrix is the transpose:

\[
M[\text{receiver},\text{source}]=\text{CSV}[\text{source},\text{receiver}].
\]

In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {CSV_PATH}. Put mij_matrix.csv in the same folder as this notebook, "
        "or update CSV_PATH."
    )

prep = nlt.load_source_receiver_csv(
    CSV_PATH,
    normalization=NORMALIZATION,
    target_spectral_radius=TARGET_SPECTRAL_RADIUS,
)

M = prep.M
df_source_receiver = prep.df_source_receiver
cell_labels = prep.labels

print("=" * 90)
print("RAW CSV CONVENTION")
print("=" * 90)
print("Rows    = outgoing/source cell types")
print("Columns = receiver/target cell types")
print(f"Raw CSV shape: {df_source_receiver.shape}")
display(df_source_receiver.iloc[:8, :8])

print("\n" + "=" * 90)
print("STATE MATRIX CONVENTION")
print("=" * 90)
print("M[receiver, source] is used in x[t+1] = M @ x[t]")
print(f"M shape: {M.shape}")
print(f"Normalization: {prep.normalization}")
print(f"Notes: {prep.notes}")
print(f"Spectral radius rho(M): {prep.spectral_radius:.6f}")

print("\nTop-left block of oriented M:")
display(pd.DataFrame(M, index=cell_labels, columns=cell_labels).iloc[:8, :8])

## 3. Quick Spectral Diagnostics

These checks clarify whether the discrete-time system is stable and how strong global recurrence is.

For

\[
x_{t+1}=Mx_t,
\]

asymptotic stability requires:

\[
\rho(M)<1.
\]

In [ ]:
eigvals = np.linalg.eigvals(M)
rho = np.max(np.abs(eigvals))
spectral_abscissa_M = np.max(np.real(eigvals))

print("=" * 90)
print("SPECTRAL DIAGNOSTICS")
print("=" * 90)
print(f"Spectral radius rho(M): {rho:.6f}")
print(f"Discrete-time stable iff rho(M)<1: {rho < 1}")
print(f"Max real eigenvalue part: {spectral_abscissa_M:.6f}")
print(f"Frobenius norm ||M||_F: {np.linalg.norm(M, 'fro'):.6f}")
print(f"Spectral norm ||M||_2: {np.linalg.norm(M, 2):.6f}")
print(f"Rank estimate: {np.linalg.matrix_rank(M)} / {M.shape[0]}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(eigvals.real, eigvals.imag, alpha=0.75, edgecolors="black")
theta = np.linspace(0, 2*np.pi, 400)
ax.plot(np.cos(theta), np.sin(theta), linestyle="--", label="Unit circle")
ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)
ax.set_title("Eigenvalue Spectrum of M")
ax.set_xlabel("Real part")
ax.set_ylabel("Imaginary part")
ax.set_aspect("equal", adjustable="box")
ax.grid(True, linestyle=":")
ax.legend()
plt.tight_layout()
plt.show()

# Part I — Structural Path Analysis: Matrix Power Series

For the oriented matrix \(M\),

\[
(M^k)_{ij}
\]

is the total weighted contribution of all length-\(k\) walks from source node \(j\) to receiver node \(i\).

So:

- \(M\) gives direct one-step paths.
- \(M^2\) gives two-step paths.
- \(M^3\) gives three-step paths.
- \(\sum_{k=1}^{K}\alpha^k M^k\) gives a truncated weighted path reachability matrix.

This section prints the intermediate powers and top source→receiver paths at each length.

In [ ]:
path_results = nlt.matrix_power_series(
    M,
    max_power=MAX_PATH_LENGTH,
    alpha=PATH_ALPHA,
    include_identity=False,
)

powers = path_results["powers"]
cumulative_paths = path_results["cumulative"]
power_norms = path_results["norms"]

print("=" * 90)
print("MATRIX POWER SERIES SUMMARY")
print("=" * 90)
print(f"Computed powers: {list(powers.keys())}")
print(f"Cumulative matrix: sum alpha^k M^k, k=1..{MAX_PATH_LENGTH}")
print(f"alpha = {PATH_ALPHA}")

display(power_norms)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(power_norms["k"], power_norms["fro_norm"], marker="o", label="Frobenius norm")
ax.plot(power_norms["k"], power_norms["spectral_norm"], marker="s", label="Spectral norm")
ax.plot(power_norms["k"], power_norms["max_abs_entry"], marker="^", label="Max |entry|")
ax.set_title("Growth/Decay of Matrix Powers")
ax.set_xlabel("Path length k")
ax.set_ylabel("Magnitude")
ax.grid(True, linestyle=":")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Enumerate Top Paths at Each Length

Each table below lists the strongest aggregate source→receiver contributions in \(M^k\).

Remember:

\[
(M^k)[\text{receiver},\text{source}].
\]

In [ ]:
top_paths_by_k = nlt.enumerate_power_top_paths(
    powers,
    labels=cell_labels,
    top_n=TOP_N_PRINT,
    include_diagonal=False,
)

for k, df_top in top_paths_by_k.items():
    print("\n" + "=" * 90)
    print(f"TOP {TOP_N_PRINT} SOURCE→RECEIVER CONTRIBUTIONS FOR PATH LENGTH k={k}")
    print("=" * 90)
    display(df_top[["source", "receiver", "value", "abs_value"]])

## 5. Cumulative Path Reachability

This matrix summarizes paths up to the chosen maximum length:

\[
R_K=\sum_{k=1}^{K}\alpha^kM^k.
\]

Large columns indicate strong multi-step outgoing influence.

Large rows indicate strong multi-step incoming reachability.

In [ ]:
path_scores = nlt.node_path_scores(cumulative_paths, cell_labels)

print("=" * 90)
print("TOP MULTI-STEP OUTGOING PATH DRIVERS")
print("=" * 90)
display(path_scores.head(TOP_N_PRINT))

print("\n" + "=" * 90)
print("TOP MULTI-STEP INCOMING PATH RECEIVERS")
print("=" * 90)
display(path_scores.sort_values("incoming_path_score", ascending=False).head(TOP_N_PRINT))

top_cumulative_entries = nlt.top_entries(
    cumulative_paths,
    labels=cell_labels,
    n=TOP_N_PRINT,
    include_diagonal=False,
)

print("\n" + "=" * 90)
print(f"TOP {TOP_N_PRINT} ENTRIES IN CUMULATIVE PATH MATRIX")
print("=" * 90)
display(top_cumulative_entries[["source", "receiver", "value", "abs_value"]])

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    np.abs(cumulative_paths),
    xticklabels=False,
    yticklabels=False,
    cmap="viridis",
    ax=ax,
)
ax.set_title(r"Absolute Cumulative Path Reachability $|\sum_{k=1}^{K}\alpha^kM^k|$")
ax.set_xlabel("Source node")
ax.set_ylabel("Receiver node")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 7))
plot_df = path_scores.head(min(TOP_N_PRINT, len(path_scores))).iloc[::-1]
ax.barh(plot_df["cell"], plot_df["outgoing_path_score"])
ax.set_title("Top Multi-Step Outgoing Path Scores")
ax.set_xlabel("sum over receivers of |cumulative path weight|")
plt.tight_layout()
plt.show()

# Part II — Controllability Gramian Analysis: State Diffusion

For the discrete-time system

\[
x_{t+1}=Mx_t+Bu_t,
\]

the finite-horizon controllability Gramian is

\[
W_c(K)=\sum_{k=0}^{K-1}M^kBB^\mathsf{T}(M^\mathsf{T})^k.
\]

Interpretation:

- \(B\) selects which nodes receive external input.
- The diagonal of \(W_c\) measures how much each state coordinate can be energized over the horizon.
- Large off-diagonal entries indicate correlated state diffusion.
- The rank and eigenvalues of \(W_c\) indicate how many state directions are reachable.

This section uses a finite-horizon Gramian, so it works even if \(\rho(M)\geq1\).

In [ ]:
B, driver_indices = nlt.make_B(cell_labels, DRIVER_NODES)

driver_labels = [cell_labels[i] for i in driver_indices]
print("=" * 90)
print("INPUT MATRIX B")
print("=" * 90)
print(f"B shape: {B.shape}")
print(f"Number of driver channels: {len(driver_indices)}")
print("First driver labels:")
for label in driver_labels[:min(20, len(driver_labels))]:
    print("  -", label)

gramian_discrete = nlt.controllability_gramian_discrete(
    M,
    B,
    horizon=GRAMIAN_HORIZON,
)

Wc_d = gramian_discrete["Wc"]
gramian_steps = gramian_discrete["increments"]
gramian_scores = nlt.controllability_scores(Wc_d, cell_labels)

print("\n" + "=" * 90)
print("DISCRETE FINITE-HORIZON CONTROLLABILITY GRAMIAN")
print("=" * 90)
print(f"Horizon K: {GRAMIAN_HORIZON}")
print(f"trace(Wc): {gramian_discrete['trace']:.6f}")
print(f"rank estimate: {gramian_discrete['rank_est']} / {M.shape[0]}")
print(f"condition number: {gramian_discrete['condition_number']:.6e}")

print("\nIntermediate growth of Gramian:")
display(gramian_steps.head(15))
display(gramian_steps.tail(5))

print("\nTop controllable/state-diffusion cells by diagonal Wc:")
display(gramian_scores.head(TOP_N_PRINT))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(gramian_steps["k"], gramian_steps["term_trace"], marker="o", label="term trace")
ax.plot(gramian_steps["k"], gramian_steps["cumulative_trace"], marker="s", label="cumulative trace")
ax.set_title("Controllability Gramian Accumulation")
ax.set_xlabel("k")
ax.set_ylabel("Trace")
ax.grid(True, linestyle=":")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    np.abs(Wc_d),
    xticklabels=False,
    yticklabels=False,
    cmap="magma",
    ax=ax,
)
ax.set_title(r"Discrete Finite-Horizon Controllability Gramian $|W_c|$")
ax.set_xlabel("State coordinate")
ax.set_ylabel("State coordinate")
plt.tight_layout()
plt.show()

eig_W = np.maximum(gramian_discrete["eigvals"], 0)
fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(np.sort(eig_W)[::-1] + 1e-18, marker="o")
ax.set_title("Controllability Gramian Eigenvalue Spectrum")
ax.set_xlabel("Eigenvalue rank")
ax.set_ylabel("Eigenvalue")
ax.grid(True, linestyle=":")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 7))
plot_df = gramian_scores.head(min(TOP_N_PRINT, len(gramian_scores))).iloc[::-1]
ax.barh(plot_df["cell"], plot_df["diagonal_energy"])
ax.set_title("Top State-Diffusion Scores from diag(Wc)")
ax.set_xlabel("diag(Wc)")
plt.tight_layout()
plt.show()

## 6. Continuous-Time Gramian Companion

The transfer-function section uses a continuous-time system:

\[
\dot{x}=Ax+Bu,\qquad y=Cx.
\]

Because the raw structural matrix \(M\) is not necessarily a stable continuous-time generator, the notebook constructs

\[
A=M-\gamma I
\]

where \(\gamma\) shifts all eigenvalues left enough to make \(A\) stable.

This does **not** change off-diagonal structural pathways, but it adds uniform leakage/decay.

In [ ]:
stable_info = nlt.make_stable_continuous_A(
    M,
    margin=CONTINUOUS_SHIFT_MARGIN,
)

A = stable_info["A"]

print("=" * 90)
print("CONTINUOUS-TIME A MATRIX CONSTRUCTION")
print("=" * 90)
print("A = M - shift * I")
print(f"spectral abscissa of M: {stable_info['spectral_abscissa_M']:.6f}")
print(f"shift used: {stable_info['shift']:.6f}")
print(f"spectral abscissa of A: {stable_info['spectral_abscissa_A']:.6f}")
print(f"Continuous-time stable iff max Re(lambda(A)) < 0: {stable_info['spectral_abscissa_A'] < 0}")

gramian_continuous = nlt.controllability_gramian_continuous(A, B)
Wc_c = gramian_continuous["Wc"]
continuous_scores = nlt.controllability_scores(Wc_c, cell_labels)

print("\n" + "=" * 90)
print("CONTINUOUS INFINITE-HORIZON CONTROLLABILITY GRAMIAN")
print("=" * 90)
print(f"trace(Wc): {gramian_continuous['trace']:.6f}")
print(f"rank estimate: {gramian_continuous['rank_est']} / {M.shape[0]}")
print(f"condition number: {gramian_continuous['condition_number']:.6e}")

display(continuous_scores.head(TOP_N_PRINT))

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    np.abs(Wc_c),
    xticklabels=False,
    yticklabels=False,
    cmap="magma",
    ax=ax,
)
ax.set_title(r"Continuous-Time Controllability Gramian $|W_c|$")
ax.set_xlabel("State coordinate")
ax.set_ylabel("State coordinate")
plt.tight_layout()
plt.show()

# Part III — Frequency Response

For the continuous-time system

\[
\dot{x}=Ax+Bu,\qquad y=Cx,
\]

the transfer function is

\[
G(s)=C(sI-A)^{-1}B.
\]

At \(s=i\omega\), the response magnitude shows which frequencies are amplified by the network.

This section computes:

\[
\|G(i\omega)\|_2
\]

and prints the strongest input→output channels at the peak frequency.

In [ ]:
C, output_indices = nlt.make_C(cell_labels, OUTPUT_NODES)

output_labels = [cell_labels[i] for i in output_indices]

omega = np.logspace(np.log10(FREQ_MIN), np.log10(FREQ_MAX), FREQ_POINTS)

freq_results = nlt.frequency_response(A, B, C, omega)
gains_2 = freq_results["gains_2norm"]
gains_fro = freq_results["gains_fro"]

peak_idx = int(np.argmax(gains_2))
peak_omega = float(omega[peak_idx])
peak_gain = float(gains_2[peak_idx])
peak_G = freq_results["responses"][peak_idx]

print("=" * 90)
print("FREQUENCY RESPONSE SUMMARY")
print("=" * 90)
print(f"A shape: {A.shape}")
print(f"B shape: {B.shape}")
print(f"C shape: {C.shape}")
print(f"Frequency grid: {FREQ_MIN:g} to {FREQ_MAX:g} rad/time, {FREQ_POINTS} points")
print(f"Peak ||G(iω)||₂: {peak_gain:.6f}")
print(f"Peak ω: {peak_omega:.6g}")

freq_table = pd.DataFrame({
    "omega": omega,
    "gain_2norm": gains_2,
    "gain_fro": gains_fro,
    "min_sigma_iwI_minus_A": freq_results["min_sigma_sI_minus_A"],
})
display(freq_table.iloc[[0, 1, 2, peak_idx, -3, -2, -1]].drop_duplicates())

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(omega, gains_2, label=r"$||G(i\omega)||_2$")
ax.loglog(omega, gains_fro, linestyle="--", label=r"$||G(i\omega)||_F$")
ax.axvline(peak_omega, linestyle=":", label=f"peak ω={peak_omega:.3g}")
ax.set_title("Frequency Response Magnitude")
ax.set_xlabel(r"Angular frequency $\omega$")
ax.set_ylabel("Gain")
ax.grid(True, which="both", linestyle=":")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Top Transfer Channels at Peak Frequency

The transfer matrix has convention:

\[
G[\text{output},\text{input}].
\]

So the table below reads as:

\[
\text{input node} \rightarrow \text{output node}.
\]

In [ ]:
top_channels = nlt.top_frequency_channels(
    peak_G,
    input_labels=driver_labels,
    output_labels=output_labels,
    n=TOP_N_PRINT,
)

print("=" * 90)
print(f"TOP {TOP_N_PRINT} TRANSFER CHANNELS AT PEAK FREQUENCY")
print("=" * 90)
print(f"Peak omega: {peak_omega:.6g}")
display(top_channels[["input", "output", "magnitude", "phase_degrees", "value"]])

# Visualize a small top-left block if all-to-all is large.
max_show = min(40, peak_G.shape[0], peak_G.shape[1])
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    np.abs(peak_G[:max_show, :max_show]),
    xticklabels=driver_labels[:max_show],
    yticklabels=output_labels[:max_show],
    cmap="cividis",
    ax=ax,
)
ax.set_title(f"|G(iω)| at peak frequency, first {max_show} outputs × inputs")
ax.set_xlabel("Input channel")
ax.set_ylabel("Output channel")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Optional: Frequency Response at Selected Channels

Use this section to track specific input→output pairs across frequency.

Edit `CHANNELS_TO_TRACE` with labels or integer indices.

In [ ]:
# Examples:
# CHANNELS_TO_TRACE = [
#     ("Some input label", "Some output label"),
#     (0, 0),
# ]
CHANNELS_TO_TRACE = []

label_to_driver_pos = {label: pos for pos, label in enumerate(driver_labels)}
label_to_output_pos = {label: pos for pos, label in enumerate(output_labels)}

if len(CHANNELS_TO_TRACE) == 0:
    print("No channels selected. Add pairs to CHANNELS_TO_TRACE to plot specific transfer entries.")
else:
    fig, ax = plt.subplots(figsize=(9, 5))
    for input_node, output_node in CHANNELS_TO_TRACE:
        in_pos = input_node if isinstance(input_node, int) else label_to_driver_pos[input_node]
        out_pos = output_node if isinstance(output_node, int) else label_to_output_pos[output_node]

        channel_gain = np.array([
            abs(G[out_pos, in_pos]) for G in freq_results["responses"]
        ])

        ax.loglog(
            omega,
            channel_gain,
            label=f"{driver_labels[in_pos]} → {output_labels[out_pos]}",
        )

    ax.set_title("Selected Transfer-Function Channels")
    ax.set_xlabel(r"Angular frequency $\omega$")
    ax.set_ylabel(r"$|G_{output,input}(i\omega)|$")
    ax.grid(True, which="both", linestyle=":")
    ax.legend()
    plt.tight_layout()
    plt.show()

# Final Interpretation Checklist

Use this checklist when reading the results:

1. **Matrix powers** answer: “Who reaches whom through paths of length \(k\)?”
2. **Cumulative path reachability** answers: “Who has the strongest total multi-step structural influence?”
3. **Discrete controllability Gramian** answers: “Which states can be energized over a finite number of update steps?”
4. **Continuous Gramian** answers: “Which states diffuse energy under a stable leakage model?”
5. **Frequency response** answers: “Which input-output channels are amplified at which frequencies?”

The notebook intentionally prints intermediate objects and top-ranked tables so you can audit every step.